In [6]:
import os
import pandas as pd
import numpy as np
import mlflow

from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("="*80)
print("            ONLINE RETAIL II PROJECT CHECKPOINT")
print("="*80)

# ==========================================================
# Load datasets
# ==========================================================

clean_df = pd.read_csv("../Data/online_retail_cleaned.csv")
feature_df = pd.read_csv("../Data/online_retail_feature_engineered.csv")
segment_df = pd.read_csv("../Data/customer_segmented.csv")
forecast_df = pd.read_csv("../Data/prophet_forecast.csv")

# ==========================================================
# Dataset Summary
# ==========================================================

print("\nDATASET SUMMARY")
print("-"*80)

print(f"Clean Dataset Shape             : {clean_df.shape}")
print(f"Feature Dataset Shape           : {feature_df.shape}")
print(f"Customer Dataset Shape          : {segment_df.shape}")
print(f"Forecast Dataset Shape          : {forecast_df.shape}")

# ==========================================================
# EDA SUMMARY
# ==========================================================

print("\nEDA")
print("-"*80)

print(f"Rows                           : {clean_df.shape[0]}")
print(f"Columns                        : {clean_df.shape[1]}")
print(f"Missing Values                 : {clean_df.isnull().sum().sum()}")
print(f"Duplicate Rows                 : {clean_df.duplicated().sum()}")

# ==========================================================
# CUSTOMER SEGMENTATION
# ==========================================================

rfm = segment_df[
    ["CustomerID","Recency","Frequency","Monetary"]
].drop_duplicates()

X = StandardScaler().fit_transform(
    rfm[["Recency","Frequency","Monetary"]]
)

kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

clusters = kmeans.fit_predict(X)

sil_score = silhouette_score(X, clusters)

print("\nCUSTOMER SEGMENTATION")
print("-"*80)

print("Algorithm                      : KMeans")
print(f"Clusters                       : {len(np.unique(clusters))}")
print(f"Silhouette Score               : {sil_score:.4f}")

# ==========================================================
# Customer Types
# ==========================================================

print("\nCustomer Groups")

if "CustomerSegment" in segment_df.columns:

    for s in segment_df["CustomerSegment"].value_counts().index:

        c = segment_df["CustomerSegment"].value_counts()[s]

        print(f"{s:30} {c}")

# ==========================================================
# Prophet Evaluation
# ==========================================================

print("\nPROPHET FORECAST")
print("-"*80)

if {"y","yhat"}.issubset(forecast_df.columns):

    mae = mean_absolute_error(
        forecast_df["y"],
        forecast_df["yhat"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            forecast_df["y"],
            forecast_df["yhat"]
        )
    )

    mape = np.mean(
        np.abs(
            (forecast_df["y"]-forecast_df["yhat"])
            /forecast_df["y"]
        )
    )*100

    print(f"MAE                           : {mae:.2f}")
    print(f"RMSE                          : {rmse:.2f}")
    print(f"MAPE                          : {mape:.2f}%")

else:

    mae = None
    rmse = None
    mape = None

    print("Ground truth column 'y' not found.")
    print("Only future forecast available.")

# ==========================================================
# MLflow
# ==========================================================

mlflow.set_experiment("Online Retail II")

with mlflow.start_run():

    mlflow.log_param("Dataset","Online Retail II")

    mlflow.log_param("Segmentation","KMeans")

    mlflow.log_param("Clusters",5)

    mlflow.log_metric(
        "Silhouette Score",
        sil_score
    )

    if mae is not None:
        mlflow.log_metric("MAE",mae)
        mlflow.log_metric("RMSE",rmse)
        mlflow.log_metric("MAPE",mape)

    for file in [
        "online_retail_cleaned.csv",
        "online_retail_feature_engineered.csv",
        "customer_segmented.csv",
        "prophet_forecast.csv",
        "lstm_forecast.csv"
    ]:
        if os.path.exists(file):
            mlflow.log_artifact(file)

    run_id = mlflow.active_run().info.run_id

print("\nMLFLOW")
print("-"*80)
print("Experiment                    : Online Retail II")
print(f"Run ID                        : {run_id}")

# ==========================================================
# Pipeline Status
# ==========================================================

print("\nPIPELINE STATUS")
print("-"*80)

steps = [
    "EDA",
    "Data Cleaning",
    "Great Expectations Validation",
    "ETL Pipeline",
    "Feature Engineering",
    "Customer Segmentation",
    "Time Series Preparation",
    "Prophet Forecasting",
    "LSTM Forecasting"
]

for step in steps:
    print(f"✔ {step}")

print("\n"+"="*80)
print("CHECKPOINT COMPLETED SUCCESSFULLY")
print("="*80)

            ONLINE RETAIL II PROJECT CHECKPOINT

DATASET SUMMARY
--------------------------------------------------------------------------------
Clean Dataset Shape             : (663373, 12)
Feature Dataset Shape           : (663373, 29)
Customer Dataset Shape          : (663373, 30)
Forecast Dataset Shape          : (31466, 19)

EDA
--------------------------------------------------------------------------------
Rows                           : 663373
Columns                        : 12
Missing Values                 : 0
Duplicate Rows                 : 0

CUSTOMER SEGMENTATION
--------------------------------------------------------------------------------
Algorithm                      : KMeans
Clusters                       : 5
Silhouette Score               : 0.5787

Customer Groups
Premium Customers              346107
New Customers                  201640
Regular Customers              70291
High Value Customers           36518
Occasional Customers           8817

PROPHET FORE